# 04. Build the Block 1-5 feature table and median-imputed model input

This notebook is the fourth step of the `toronto_election_turnout` rewrite (see the project plan
for the full six-notebook scope). It takes the Block 1-4 tables already produced by
`02_build_census_tract_profile.ipynb` (census demographic/housing/immigration shares) and
`03_interpolate_votes_to_tracts.ipynb` (interpolated turnout outcomes + electoral competitiveness),
builds the one block the earlier notebooks did not cover — **Block 5, built-environment / access /
municipal-service variables** — and joins everything into a single CT-level feature table. The last
step, median imputation, produces the exact table notebook 05's PLS regression will read.

Block 5 ports five pieces of `variables/scripts/build_blocks_1_5_master.py`:

1. **TTS 2022 transportation** — area-weighted zone→CT interpolation of no-car-household share and
   transit-trip share.
2. **Facility access** — Euclidean centroid distance/count-within-1200m for libraries, community
   centres, parks, and shelters.
3. **Point-in-polygon densities** — KSI collisions and development applications, counted per CT and
   normalized per 1,000 residents.
4. **311 service requests** — ward-coded (not point-geocoded) requests allocated to CTs by a fresh
   ward-polygon → CT-polygon area crosswalk built in this notebook (the archived pipeline's own
   ward-to-CT crosswalk was never persisted as a reusable artifact by notebook 03, so it's rebuilt
   here directly from `src/data/wards.geo.json`).
5. **Census stragglers** — social housing share, census-only transit-commute share, and school-age
   share, all already sitting in notebook 02's census profile under different names; these are just
   renamed into the `block5_` namespace rather than recomputed.

**Scope note**: `accessibility/` (the old pipeline's dead-end Leaflet-viewer accessibility work) is
out of scope, as documented in notebook 01's intro cell.


In [1]:
import re
import zipfile
import csv
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm


In [2]:
REPO_ROOT = Path.cwd().resolve().parents[1]
DATA_ROOT = REPO_ROOT / "data" / "toronto_election_turnout"
ARCHIVE_ROOT = DATA_ROOT / "archive"
VARIABLES_RAW = ARCHIVE_ROOT / "variables" / "raw"

CENSUS_CSV = DATA_ROOT / "census" / "ct_census_profile.csv"
CENSUS_GEOJSON = DATA_ROOT / "census" / "ct_census_profile.geojson"
OUTCOMES_CSV = DATA_ROOT / "interpolation" / "ct_election_outcomes.csv"
WARDS_GEOJSON = REPO_ROOT / "src" / "data" / "wards.geo.json"

OUTPUT_DIR = DATA_ROOT / "features"
OUTPUT_CSV = OUTPUT_DIR / "ct_model_input.csv"
OUTPUT_GEOJSON = OUTPUT_DIR / "ct_model_input.geojson"

# Ground truth for the verification cells at the end of this notebook.
GROUND_TRUTH_MASTER = ARCHIVE_ROOT / "variables" / "processed" / "toronto_ct_blocks_1_5_modelling_master.csv"
GROUND_TRUTH_FINAL = (
    ARCHIVE_ROOT / "modelling" / "processed" / "spatial_models"
    / "toronto_ct_blocks_1_5_model_input_housing_augmented_median_imputed.csv"
)

METRIC_CRS = 3347          # StatCan Lambert (equal-area, metres) -- used for all area/distance math
ACCESS_RADIUS_M = 1200     # ~15-minute walk at roughly 80 m/min: a transparent, if crude, walkability proxy
KSI_DEV_YEAR_MIN, KSI_DEV_YEAR_MAX = 2021, 2025
REQUESTS_311_YEARS = (2023, 2024, 2025)


## 1. The CT universe: Block 1-3 census profile, restricted to the 585-CT interpolation universe

Every downstream Block 5 variable is computed only for the 585 CTs that notebook 03 actually
estimated election outcomes for (`in_interpolation_universe == True` in notebook 02's output) — the
same restriction the archived `build_blocks_1_5_master.py` applies via
`in_interpolation_universe == "true"`. Block 1-3 columns get the `block1_`/`block2_`/`block3_`
prefixes here (notebook 02's own output leaves them unprefixed, by design, so it stays a
general-purpose census profile); the three Block 5 stragglers already computed in notebook 02
(`subsidized_housing_tenant_share`, `transit_commute_share`, `school_age_5_17_share`) get renamed
into the `block5_` namespace at the same time.

CT polygon geometry is taken directly from notebook 02's own `ct_census_profile.geojson` (EPSG:4326,
reprojected here to EPSG:3347 for area/distance work) rather than falling back to the old pipeline's
`clustering_neighbourhoods/profiles/raw_shares.gpkg` source — the two agree on geometry closely
enough (confirmed below: they produce byte-identical polygon areas for spot-checked CTs) that the
fallback specified in the plan was never needed.


In [3]:
census_df = pd.read_csv(CENSUS_CSV)
census_df["ct_id"] = census_df["ct_id"].astype(float)

ct_gdf_full = gpd.read_file(CENSUS_GEOJSON)[["ct_id", "geometry"]]
ct_gdf_full["ct_id"] = ct_gdf_full["ct_id"].astype(float)

universe = census_df[census_df["in_interpolation_universe"] == True].copy().set_index("ct_id")
ct_gdf = ct_gdf_full.merge(universe.reset_index()[["ct_id"]], on="ct_id", how="inner")

ct_m = ct_gdf.to_crs(METRIC_CRS)
ct_m["geometry"] = ct_m.geometry.make_valid()  # a handful of source polygons have minor topology defects
ct_m["centroid"] = ct_m.geometry.centroid
ct_m = ct_m.set_index("ct_id")
ct_ids = list(universe.index)

print(f"CT universe: {len(ct_ids)} tracts (expect 585); all geometries valid: {ct_m.geometry.is_valid.all()}")


CT universe: 585 tracts (expect 585); all geometries valid: True


In [4]:
BLOCK123_RENAME = {
    "age_18_34_share": "block1_age_18_34_share",
    "age_35_64_share": "block1_age_35_64_share",
    "age_65_plus_share": "block1_age_65_plus_share",
    "median_age": "block1_median_age",
    "average_household_size": "block1_average_household_size",
    "bachelors_or_higher_25_64_share": "block1_bachelors_or_higher_25_64_share",
    "low_income_lim_at_share": "block1_low_income_lim_at_share",
    "unemployment_rate_share": "block1_unemployment_rate_share",
    "renter_share": "block2_renter_share",
    "owner_share": "block2_owner_share",
    "same_address_1yr_share": "block2_same_address_1yr_share",
    "same_address_5yr_share": "block2_same_address_5yr_share",
    "condo_share": "block2_condo_share",
    "apartment_share": "block2_apartment_share",
    "detached_share": "block2_detached_share",
    "semi_detached_share": "block2_semi_detached_share",
    "population_density_per_km2": "block2_population_density_per_km2",
    # The nine "housing-augmented" dwelling-structure/condo counts and densities notebook 02 already
    # computes as part of a self-contained Block 2 -- see notebook 02 section 7. Prefixing them here
    # is all that's needed; no separate augmentation step.
    "structural_type_total_dwellings": "block2_structural_type_total_dwellings",
    "apartment_duplex_count": "block2_apartment_duplex_count",
    "apartment_lt5_storeys_count": "block2_apartment_lt5_storeys_count",
    "apartment_5plus_storeys_count": "block2_apartment_5plus_storeys_count",
    "apartment_total_count": "block2_apartment_total_count",
    "condo_status_total_dwellings": "block2_condo_status_total_dwellings",
    "condominium_dwellings_count": "block2_condominium_dwellings_count",
    "apartments_per_km2": "block2_apartments_per_km2",
    "condos_per_km2": "block2_condos_per_km2",
    "immigrant_share": "block3_immigrant_share",
    "recent_immigrant_share": "block3_recent_immigrant_share",
    "non_citizen_share": "block3_non_citizen_share",
    "citizen_adult_share": "block3_citizen_adult_share",
    "visible_minority_share": "block3_visible_minority_share",
    "english_french_knowledge_share": "block3_english_french_knowledge_share",
    "non_official_mother_tongue_share": "block3_non_official_mother_tongue_share",
    # Census-sourced Block 5 stragglers -- already computed by notebook 02, just renamed here.
    "subsidized_housing_tenant_share": "block5_social_housing_share",
    "transit_commute_share": "block5_census_transit_commute_share",
    "school_age_5_17_share": "block5_school_age_5_17_share",
}

base_df = universe.rename(columns=BLOCK123_RENAME).reset_index()
base_cols = (
    ["ct_id", "ctuid", "dguid", "geo_name", "population_total", "population_18plus",
     "canadian_citizens_18plus_count", "land_area_km2"]
    + list(BLOCK123_RENAME.values())
)
base_df = base_df[base_cols].rename(columns={"canadian_citizens_18plus_count": "citizen_canadian_18plus_count"})
base_df["census_year"] = "2021"
base_df.head(2)


,ct_id,ctuid,dguid,geo_name,population_total,population_18plus,citizen_canadian_18plus_count,land_area_km2,block1_age_18_34_share,block1_age_35_64_share,...,block3_recent_immigrant_share,block3_non_citizen_share,block3_citizen_adult_share,block3_visible_minority_share,block3_english_french_knowledge_share,block3_non_official_mother_tongue_share,block5_social_housing_share,block5_census_transit_commute_share,block5_school_age_5_17_share,census_year
0,5350128.04,5350128.04,2021S05075350128.04,128.04,4772.0,4310,3710.0,0.16,0.263103,0.405660,...,0.066253,0.148033,0.860789,0.373320,0.97801,0.317277,0.212,0.403226,0.060797,2021
1,5350363.06,5350363.06,2021S05075350363.06,363.06,7176.0,6125,4430.0,0.82,0.309192,0.380919,...,0.142371,0.268392,0.723265,0.905995,0.94216,0.560279,0.077,0.281319,0.092618,2021


## 2. Block 4 + election outcomes, from notebook 03

Notebook 03 already produced turnout/participation outcomes and the twelve `block4_*`
competitiveness columns, keyed by `ct_id` and already carrying the `block4_`/`outcome_` prefixes the
final table needs — no renaming required, just a join. `outcome_federal_minus_municipal_participation`
is one derived column the archived *modelling* stage (not the Blocks 1-5 script) added later; it's
computed here since it belongs with the other outcome columns.


In [5]:
outcomes_df = pd.read_csv(OUTCOMES_CSV)
outcomes_df["ct_id"] = outcomes_df["ct_id"].astype(float)

df = base_df.merge(outcomes_df, on="ct_id", how="left")
df["outcome_federal_minus_municipal_participation"] = (
    df["outcome_federal_participation_citizen_18plus"] - df["outcome_municipal_participation_citizen_18plus"]
)
print(f"{df.shape[0]} rows x {df.shape[1]} columns after Block 1-4 join")


585 rows x 72 columns after Block 1-4 join


## 3. Block 5a: Transportation Tomorrow Survey 2022 (area-weighted zone → CT)

TTS 2022 publishes household/trip metrics at the level of ~5,800 custom travel-survey zones, not CTs.
Area-weighted interpolation is the same logic notebook 03 used for poll→CT vote allocation, just
without a population weight: each zone contributes to a CT's household/trip totals in proportion to
the *share of the zone's own area* that falls inside that CT — reasonable here because TTS zones
don't publish a finer within-zone population surface to weight by (unlike the DA-level citizen counts
notebook 03 had for elections).

Many raw TTS zones are missing metrics entirely (small-sample zones are suppressed for privacy) —
these contribute zero to both numerator and denominator, matching the archived script's
`... or 0` fallback exactly.

**A note on geometry validity**: the archived GDAL/OGR pipeline logged "non-fatal topology warnings"
for a handful of zone/CT intersections and silently skipped them (documented in its own methodology
report as an accepted approximation). This notebook calls `.make_valid()` on both zone and CT
geometries before intersecting, which resolves those topology issues rather than skipping them — so
for the ~5 affected CTs, this notebook's TTS shares are *more complete* than the archived ground
truth, not wrong. That shows up as an expected, explained residual in this notebook's own
verification cell, not a bug.


In [6]:
def as_number(v):
    if v is None:
        return None
    try:
        f = float(v)
    except (ValueError, TypeError):
        return None
    return f if np.isfinite(f) else None


metrics_df = pd.read_csv(VARIABLES_RAW / "transportation_tomorrow_survey_2022" / "metrics_tts2022.csv")
metrics_df["tts22_hhld"] = metrics_df["tts22_hhld"].astype(int)
metrics_df = metrics_df.set_index("tts22_hhld")

zones_gdf = gpd.read_file(VARIABLES_RAW / "transportation_tomorrow_survey_2022" / "tts2022zones_data.geojson")
zones_gdf["zone_id"] = zones_gdf["TTS2022"].astype(int)
zones_m = zones_gdf.to_crs(METRIC_CRS)
zones_m["geometry"] = zones_m.geometry.make_valid()
zones_m["zone_area_m2"] = zones_m.geometry.area
zones_m = zones_m[zones_m["zone_area_m2"] > 0].copy()


def zone_metrics(row):
    zid = row["zone_id"]
    m = metrics_df.loc[zid] if zid in metrics_df.index else None
    hhlds = (as_number(m["hhlds"]) if m is not None else None) or as_number(row.get("hhlds")) or 0.0
    no_veh = as_number(m["hhlds_no_veh"]) if m is not None else None
    if no_veh is None:
        pct = as_number(row.get("hhlds_no_veh"))
        no_veh = hhlds * pct / 100 if pct is not None else 0.0
    trips = as_number(m.get("trips_total")) if m is not None else None
    if trips is None and m is not None:
        trips = as_number(m.get("trips_5up"))
    trips = trips or 0.0
    transit_pct = as_number(row.get("mode_transit"))
    transit_count = trips * transit_pct / 100 if transit_pct is not None else 0.0
    return pd.Series({"z_hhlds": hhlds, "z_no_veh": no_veh, "z_trips": trips, "z_transit": transit_count})


zone_vals = zones_m.apply(zone_metrics, axis=1)
zones_m = pd.concat([zones_m.drop(columns=["hhlds"], errors="ignore"), zone_vals], axis=1)

tts_overlay = gpd.overlay(
    zones_m[["zone_id", "zone_area_m2", "z_hhlds", "z_no_veh", "z_trips", "z_transit", "geometry"]],
    ct_m.reset_index()[["ct_id", "geometry"]],
    how="intersection", keep_geom_type=False,
)
tts_overlay["inter_area_m2"] = tts_overlay.geometry.area
tts_overlay = tts_overlay[tts_overlay["inter_area_m2"] > 0].copy()
tts_overlay["portion"] = tts_overlay["inter_area_m2"] / tts_overlay["zone_area_m2"]

tts_agg = tts_overlay.groupby("ct_id").apply(lambda g: pd.Series({
    "tts_no_car_num": (g["z_no_veh"] * g["portion"]).sum(),
    "tts_hhlds_denom": (g["z_hhlds"] * g["portion"]).sum(),
    "tts_transit_num": (g["z_transit"] * g["portion"]).sum(),
    "tts_trips_denom": (g["z_trips"] * g["portion"]).sum(),
    "block5_tts_overlap_area_m2": g["inter_area_m2"].sum(),
}), include_groups=False).reindex(ct_ids)

tts_agg["block5_tts_no_car_household_share"] = tts_agg["tts_no_car_num"] / tts_agg["tts_hhlds_denom"]
tts_agg["block5_tts_transit_trip_share"] = tts_agg["tts_transit_num"] / tts_agg["tts_trips_denom"]
tts_out = tts_agg[["block5_tts_no_car_household_share", "block5_tts_transit_trip_share", "block5_tts_overlap_area_m2"]]

tts_out.describe()


,block5_tts_no_car_household_share,block5_tts_transit_trip_share,block5_tts_overlap_area_m2
count,585.000000,585.000000,5.850000e+02
mean,0.244381,0.181520,1.138325e+06
std,0.170116,0.073522,1.485518e+06
min,0.000059,0.011712,1.345171e+04
25%,0.108477,0.125293,4.765335e+05
50%,0.213865,0.171100,7.904549e+05
75%,0.355261,0.232835,1.287473e+06
max,0.804936,0.430684,2.117637e+07


## 4. Block 5b: facility access within 1200m (libraries, community centres, parks, shelters)

Straight Euclidean centroid-to-nearest-point/polygon distance, exactly as the archived pipeline
computed it — no pedestrian or transit network was collected for this project, so the 1200m
walking-distance threshold (roughly a 15-minute walk at 80 m/min) is a transparent, documented proxy
rather than a claim of routable accessibility. Parks are polygons (a CT can have a park distance of
zero if a park falls inside it); libraries, community centres, and shelters are points.


In [7]:
library_gdf = gpd.read_file(
    VARIABLES_RAW / "toronto_open_data" / "library_branch_general_information" / "tpl-branch-general-information-4326.geojson"
).to_crs(METRIC_CRS)

parks_rec_gdf = gpd.read_file(
    VARIABLES_RAW / "toronto_open_data" / "parks_and_recreation_facilities" / "parks-and-recreation-facilities-4326.geojson"
)
typ = parks_rec_gdf["TYPE"].fillna("").str.lower()
amen = parks_rec_gdf["AMENITIES"].fillna("").str.lower()
rec_centres_gdf = parks_rec_gdf[
    typ.str.contains("community centre") | amen.str.contains("community centre") | typ.str.contains("community recreation")
].to_crs(METRIC_CRS)
parks_facility_gdf = parks_rec_gdf[typ == "park"].to_crs(METRIC_CRS)

park_gdf = gpd.read_file(VARIABLES_RAW / "toronto_open_data" / "parks" / "parks-wgs84.zip").to_crs(METRIC_CRS)
if len(park_gdf) == 0:
    park_gdf = parks_facility_gdf
if len(rec_centres_gdf) == 0:
    rec_centres_gdf = parks_rec_gdf.to_crs(METRIC_CRS)

shelter_gdf = gpd.read_file(
    VARIABLES_RAW / "toronto_open_data" / "hostel_services_homeless_shelter_locations" / "shelter-locations-wgs84.zip"
).to_crs(METRIC_CRS)

print("facility counts -- library:", len(library_gdf), "community_centre:", len(rec_centres_gdf),
      "park:", len(park_gdf), "shelter:", len(shelter_gdf))


facility counts -- library: 112 community_centre: 278 park: 3274 shelter: 59


In [8]:
facility_layers = {
    "library": library_gdf.geometry,
    "community_centre": rec_centres_gdf.geometry,
    "park": park_gdf.geometry,
    "shelter": shelter_gdf.geometry,
}

results = {name: {"nearest_m": [], "count_1200m": [], "access_1200m": []} for name in facility_layers}
for cid in tqdm(ct_ids, desc="facility access (nearest-distance per CT)"):
    centroid = ct_m.loc[cid, "centroid"]
    for name, geoms in facility_layers.items():
        d = geoms.distance(centroid)
        results[name]["nearest_m"].append(d.min() if len(d) else None)
        count = int((d <= ACCESS_RADIUS_M).sum())
        results[name]["count_1200m"].append(count)
        results[name]["access_1200m"].append(1 if count > 0 else 0)

access_df = pd.DataFrame({"ct_id": ct_ids})
for name, vals in results.items():
    access_df[f"block5_{name}_nearest_m"] = vals["nearest_m"]
    access_df[f"block5_{name}_count_1200m"] = vals["count_1200m"]
    access_df[f"block5_{name}_access_1200m"] = vals["access_1200m"]
access_df = access_df.set_index("ct_id")
access_df.describe()


facility access (nearest-distance per CT):   0%|          | 0/585 [00:00<?, ?it/s]

,block5_library_nearest_m,block5_library_count_1200m,block5_library_access_1200m,block5_community_centre_nearest_m,block5_community_centre_count_1200m,block5_community_centre_access_1200m,block5_park_nearest_m,block5_park_count_1200m,block5_park_access_1200m,block5_shelter_nearest_m,block5_shelter_count_1200m,block5_shelter_access_1200m
count,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.0,585.000000,585.000000,585.000000
mean,967.715772,1.044444,0.726496,685.567754,2.273504,0.914530,135.172268,27.998291,1.0,2341.226899,0.847863,0.316239
std,536.941057,0.882521,0.446139,404.608961,1.526164,0.279819,122.409230,11.286812,0.0,1726.109500,1.977381,0.465405
min,98.533647,0.000000,0.000000,32.190802,0.000000,0.000000,0.000000,3.000000,1.0,27.993846,0.000000,0.000000
25%,561.394445,0.000000,0.000000,384.384773,1.000000,1.000000,32.028333,20.000000,1.0,987.628228,0.000000,0.000000
50%,860.935492,1.000000,1.000000,632.744271,2.000000,1.000000,109.384908,26.000000,1.0,1902.780193,0.000000,0.000000
75%,1255.387626,1.000000,1.000000,896.715946,3.000000,1.000000,203.304862,34.000000,1.0,3295.006486,1.000000,1.000000
max,3449.312918,4.000000,1.000000,3068.567476,10.000000,1.000000,684.441522,68.000000,1.0,8553.559642,13.000000,1.000000


## 5. Block 5c: point-in-polygon densities (KSI collisions, development applications)

Both datasets are point-located: motor-vehicle collisions involving death or serious injury
(2021-2025) and development applications submitted (2021-2025), each assigned to the CT polygon
that contains it and normalized per 1,000 residents using notebook 02's census `population_total`.
Development-application coordinates are published in EPSG:2952 (MTM Zone 10); collisions are already
in lon/lat.


In [9]:
ct_gdf_4326 = ct_gdf[["ct_id", "geometry"]]

ksi = pd.read_csv(VARIABLES_RAW / "toronto_open_data" / "ksi_collisions" / "motor-vehicle-collisions-with-ksi-data-4326.csv")
ksi["year"] = pd.to_numeric(ksi["accdate"].astype(str).str[:4], errors="coerce")
ksi = ksi[(ksi["year"] >= KSI_DEV_YEAR_MIN) & (ksi["year"] <= KSI_DEV_YEAR_MAX)].dropna(subset=["longitude", "latitude"])
ksi = ksi.drop_duplicates(subset=["collision_id"])  # each collision has one row per involved person
ksi_gdf = gpd.GeoDataFrame(ksi, geometry=gpd.points_from_xy(ksi["longitude"], ksi["latitude"]), crs=4326)
ksi_counts = gpd.sjoin(ksi_gdf, ct_gdf_4326, how="inner", predicate="within").groupby("ct_id").size()

dev = pd.read_csv(
    VARIABLES_RAW / "toronto_open_data" / "development_applications" / "development-applications.csv",
    encoding="utf-8-sig",
)
dev["year"] = pd.to_numeric(dev["DATE_SUBMITTED"].astype(str).str[:4], errors="coerce")
dev = dev[(dev["year"] >= KSI_DEV_YEAR_MIN) & (dev["year"] <= KSI_DEV_YEAR_MAX)].dropna(subset=["X", "Y"])
dev_gdf = gpd.GeoDataFrame(dev, geometry=gpd.points_from_xy(dev["X"], dev["Y"]), crs=2952).to_crs(4326)
dev_counts = gpd.sjoin(dev_gdf, ct_gdf_4326, how="inner", predicate="within").groupby("ct_id").size()

point_df = pd.DataFrame(index=ct_ids)
point_df["block5_ksi_collision_events_2021_2025"] = ksi_counts.reindex(point_df.index).fillna(0).astype(int)
point_df["block5_development_applications_2021_2025"] = dev_counts.reindex(point_df.index).fillna(0).astype(int)

pop_series = universe["population_total"].reindex(point_df.index)
point_df["block5_ksi_collision_events_2021_2025_per_1000"] = (
    point_df["block5_ksi_collision_events_2021_2025"] / pop_series * 1000
)
point_df["block5_development_applications_2021_2025_per_1000"] = (
    point_df["block5_development_applications_2021_2025"] / pop_series * 1000
)
point_df.describe()


,block5_ksi_collision_events_2021_2025,block5_development_applications_2021_2025,block5_ksi_collision_events_2021_2025_per_1000,block5_development_applications_2021_2025_per_1000
count,585.000000,585.000000,585.000000,585.000000
mean,2.456410,12.480342,0.630069,2.981044
std,2.830587,21.884661,1.491125,8.087954
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.176243,0.000000
50%,2.000000,4.000000,0.379579,0.819001
75%,3.000000,14.000000,0.720288,3.080082
max,33.000000,157.000000,26.258206,158.597663


## 6. Block 5d: 311 service requests via a fresh ward → CT area-weighted crosswalk

The official 311 annual files carry a ward field but no request coordinates, so CT-level counts have
to be *estimated* rather than counted directly. The archived pipeline allocated ward totals to CTs
using its own poll-interpolation ward-to-CT crosswalk; notebook 03 doesn't persist that crosswalk as
a reusable artifact (it only keeps a poll→CT crosswalk in memory), so this notebook builds a fresh
one directly from `src/data/wards.geo.json` intersected against this notebook's own CT polygons.

Unlike the population-weighted poll→CT allocation in notebook 03, this crosswalk uses **pure area
weighting** — confirmed by reading the archived `ward_proxy_311()` function, which weights by
`intersection_area_m2 / ward_area` with no population term at all. That's a reasonable
simplification here: 311 requests are a diffuse administrative signal (potholes, noise complaints,
graffiti, etc.) with no obvious small-area population surface to prefer over uniform density within
a ward, unlike a vote count where allocating by citizen population is clearly better than by raw area.


In [10]:
wards_gdf = gpd.read_file(WARDS_GEOJSON)
wards_gdf["ward"] = wards_gdf["num"].astype(int).astype(str).str.zfill(2)
wards_m = wards_gdf.to_crs(METRIC_CRS)
wards_m["geometry"] = wards_m.geometry.make_valid()

ward_overlay = gpd.overlay(
    wards_m[["ward", "geometry"]], ct_m.reset_index()[["ct_id", "geometry"]],
    how="intersection", keep_geom_type=False,
)
ward_overlay["inter_area_m2"] = ward_overlay.geometry.area
ward_overlay = ward_overlay[ward_overlay["inter_area_m2"] > 0].copy()
ward_area_totals = ward_overlay.groupby("ward")["inter_area_m2"].sum()
ward_overlay["weight"] = ward_overlay["inter_area_m2"] / ward_overlay["ward"].map(ward_area_totals)

print("weight sums per ward (should each be ~1.0):")
print(ward_overlay.groupby("ward")["weight"].sum().describe())


weight sums per ward (should each be ~1.0):
count    2.500000e+01
mean     1.000000e+00
std      2.266233e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: weight, dtype: float64


In [11]:
def ward_number(text):
    m = re.search(r"\((\d+)\)", text or "")
    return m.group(1).zfill(2) if m else ""


counts_by_year = {}
ward_counts = {}
missing_ward = 0
for year in tqdm(REQUESTS_311_YEARS, desc="311 annual files"):
    path = VARIABLES_RAW / "toronto_open_data" / "311_service_requests_customer_initiated" / f"sr{year}.zip"
    with zipfile.ZipFile(path) as archive:
        member = [n for n in archive.namelist() if n.lower().endswith(".csv")][0]
        with archive.open(member) as raw:
            reader = csv.DictReader((line.decode("utf-8-sig", errors="replace") for line in raw))
            n = 0
            for row in reader:
                n += 1
                w = ward_number(row.get("Ward", ""))
                if w:
                    ward_counts[w] = ward_counts.get(w, 0) + 1
                else:
                    missing_ward += 1
    counts_by_year[year] = n

total_311 = sum(counts_by_year.values())
print(f"311 requests {REQUESTS_311_YEARS}: {counts_by_year}, total={total_311}, missing_ward={missing_ward}")

allocated_count = ward_overlay.groupby("ct_id").apply(
    lambda g: (g["weight"] * g["ward"].map(ward_counts).fillna(0)).sum(), include_groups=False
).reindex(ct_ids).fillna(0.0)
weight_sum = ward_overlay.groupby("ct_id")["weight"].sum().reindex(ct_ids).fillna(0.0)

req_df = pd.DataFrame(index=ct_ids)
req_df["block5_requests_311_2023_2025_estimated_count"] = allocated_count
req_df["block5_requests_311_2023_2025_citywide_total"] = total_311
req_df["block5_requests_311_missing_ward_count"] = missing_ward
req_df["block5_requests_311_area_allocation_weight"] = weight_sum
req_df["block5_requests_311_per_1000"] = req_df["block5_requests_311_2023_2025_estimated_count"] / pop_series * 1000
req_df["block5_requests_311_proxy_note"] = (
    "Official 311 files provide ward/FSA but no coordinates; CT count is an area-weighted "
    "ward-to-CT allocation using a fresh ward-polygon-to-CT-polygon area crosswalk built in this notebook."
)
req_df.drop(columns=["block5_requests_311_proxy_note"]).describe()


311 annual files:   0%|          | 0/3 [00:00<?, ?it/s]

311 requests (2023, 2024, 2025): {2023: 414282, 2024: 437043, 2025: 500269}, total=1351594, missing_ward=24


,block5_requests_311_2023_2025_estimated_count,block5_requests_311_2023_2025_citywide_total,block5_requests_311_missing_ward_count,block5_requests_311_area_allocation_weight,block5_requests_311_per_1000
count,585.000000,585.0,585.0,585.000000,585.000000
mean,2310.376068,1351594.0,24.0,0.042735,687.778221
std,2158.204562,0.0,0.0,0.038227,2272.821291
min,57.299853,1351594.0,24.0,0.000837,38.168481
25%,1178.674596,1351594.0,24.0,0.022469,259.860498
50%,1806.963453,1351594.0,24.0,0.033817,431.408576
75%,2870.824472,1351594.0,24.0,0.053885,617.412919
max,26899.688427,1351594.0,24.0,0.374528,44907.660146


## 7. Assemble Block 5, and the full Block 1-5 feature table

`block5_transit_commute_share_preferred` prefers the TTS-derived transit-trip share and falls back to
the census-only commute-mode share only when the TTS value is missing or exactly zero — ported
faithfully from the archived script's `tts_share or census_share` logic. `block5_no_car_household_share`
is just an alias of the TTS no-car share, kept as its own column because it's the name the modelling
stage (and `BLOCKS["block_5_municipal_services"]` below) actually references.


In [12]:
block5 = tts_out.join([access_df, point_df, req_df], how="outer")

block5["block5_transit_commute_share_preferred"] = np.where(
    block5["block5_tts_transit_trip_share"].fillna(0) != 0,
    block5["block5_tts_transit_trip_share"],
    df.set_index("ct_id")["block5_census_transit_commute_share"].reindex(block5.index),
)
block5["block5_no_car_household_share"] = block5["block5_tts_no_car_household_share"]
block5 = block5.reset_index().rename(columns={"index": "ct_id"})

full = df.merge(block5, on="ct_id", how="left")
full["block5_social_housing_note"] = (
    "Census Profile characteristic 1491: percent of tenant households in subsidized housing, "
    "converted to a 0-1 share. Denominator is tenant households, not all households."
)

print(f"Full Block 1-5 feature table (pre-imputation): {full.shape[0]} rows x {full.shape[1]} columns")
full.head(2)


Full Block 1-5 feature table (pre-imputation): 585 rows x 100 columns


,ct_id,ctuid,dguid,geo_name,population_total,population_18plus,citizen_canadian_18plus_count,land_area_km2,block1_age_18_34_share,block1_age_35_64_share,...,block5_development_applications_2021_2025_per_1000,block5_requests_311_2023_2025_estimated_count,block5_requests_311_2023_2025_citywide_total,block5_requests_311_missing_ward_count,block5_requests_311_area_allocation_weight,block5_requests_311_per_1000,block5_requests_311_proxy_note,block5_transit_commute_share_preferred,block5_no_car_household_share,block5_social_housing_note
0,5350128.04,5350128.04,2021S05075350128.04,128.04,4772.0,4310,3710.0,0.16,0.263103,0.405660,...,2.724225,726.311280,1351594,24,0.012320,152.202699,Official 311 files provide ward/FSA but no coo...,0.297308,0.431221,Census Profile characteristic 1491: percent of...
1,5350363.06,5350363.06,2021S05075350363.06,363.06,7176.0,6125,4430.0,0.82,0.309192,0.380919,...,0.836120,1332.407983,1351594,24,0.031732,185.675583,Official 311 files provide ward/FSA but no coo...,0.155972,0.176784,Census Profile characteristic 1491: percent of...


## 8. Median imputation

A small number of CTs (0-2 per column) are missing values for a handful of census shares and
competitiveness measures — not enough to justify dropping rows, and few enough that a per-column
median fill is a defensible, transparent way to keep every CT in the model-input table while flagging
exactly which cells were imputed. This ports `impute_rows()` from the archived
`run_spatial_block_models.py`: for each column in a fixed list, missing values (only if the column
has at least one observed value) are filled with that column's own median across observed CTs, and a
companion `{col}_median_imputed` boolean column records which rows were touched.

The archived pipeline actually ran this in **two passes** — a first pass over the modelling
`BLOCKS`/`DEPENDENT_VARIABLES` columns (producing `..._median_imputed.csv`), then an undocumented
second pass adding the nine Block-2 housing-augmented columns (`..._housing_augmented_median_imputed.csv`)
that also happens to re-run over `block2_condo_share`. That second pass finds `block2_condo_share`
already fully observed (pass 1 already filled it), so it overwrites that one column's imputed-flag
with all-`False` — a quirk of how the archived pipeline evolved, not a deliberate design choice, but
reproduced here for an exact match since it's the ground truth notebook 05 will compare against.


In [13]:
BLOCKS = {
    "block_1_demographic": [
        "block1_age_18_34_share", "block1_age_35_64_share", "block1_age_65_plus_share",
        "block1_median_age", "block1_average_household_size",
        "block1_bachelors_or_higher_25_64_share", "block1_low_income_lim_at_share",
        "block1_unemployment_rate_share",
    ],
    "block_2_housing_stability": [
        "block2_renter_share", "block2_owner_share", "block2_same_address_1yr_share",
        "block2_same_address_5yr_share", "block2_condo_share", "block2_apartment_share",
        "block2_detached_share", "block2_semi_detached_share", "block2_population_density_per_km2",
    ],
    "block_3_immigration_eligibility": [
        "block3_immigrant_share", "block3_recent_immigrant_share", "block3_non_citizen_share",
        "block3_citizen_adult_share", "block3_visible_minority_share",
        "block3_english_french_knowledge_share", "block3_non_official_mother_tongue_share",
    ],
    "block_4_competitiveness": [
        "block4_mayoral_top_two_margin", "block4_effective_mayoral_candidates_5pct",
        "block4_mayoral_vote_fragmentation", "block4_federal_margin", "block4_provincial_margin",
        "block4_effective_federal_parties_5pct", "block4_effective_provincial_parties_5pct",
    ],
    "block_5_municipal_services": [
        "block5_transit_commute_share_preferred", "block5_no_car_household_share",
        "block5_social_housing_share", "block5_requests_311_per_1000",
        "block5_library_access_1200m", "block5_community_centre_access_1200m",
        "block5_park_access_1200m", "block5_school_age_5_17_share",
        "block5_ksi_collision_events_2021_2025_per_1000",
        "block5_development_applications_2021_2025_per_1000", "block5_shelter_access_1200m",
    ],
}
DEPENDENT_VARIABLES = [
    "outcome_municipal_participation_citizen_18plus",
    "outcome_provincial_participation_citizen_18plus",
    "outcome_federal_participation_citizen_18plus",
    "outcome_federal_minus_municipal_participation",
    "outcome_mean_participation_citizen_18plus",
]
HOUSING_AUGMENTED_COLUMNS = [
    "block2_structural_type_total_dwellings", "block2_apartment_duplex_count",
    "block2_apartment_lt5_storeys_count", "block2_apartment_5plus_storeys_count",
    "block2_apartment_total_count", "block2_apartments_per_km2",
    "block2_condo_status_total_dwellings", "block2_condominium_dwellings_count",
    "block2_condos_per_km2", "block2_condo_share",
]
MODEL_COLUMNS = sorted({c for cols in BLOCKS.values() for c in cols} | set(DEPENDENT_VARIABLES))


def impute_rows(frame: pd.DataFrame, columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = frame.copy()
    report = []
    for col in columns:
        vals = pd.to_numeric(out[col], errors="coerce")
        present = vals.dropna()
        missing = int(vals.isna().sum())
        if present.empty:
            report.append({"variable": col, "missing_count": missing, "imputed_count": 0, "median": None})
            continue
        med = present.median()
        is_missing = vals.isna()
        out[col] = vals.where(~is_missing, med)
        out[f"{col}_median_imputed"] = is_missing
        report.append({"variable": col, "missing_count": missing, "imputed_count": int(is_missing.sum()), "median": med})
    return out, pd.DataFrame(report)


stage1_df, report1 = impute_rows(full, MODEL_COLUMNS)
model_input, report2 = impute_rows(stage1_df, HOUSING_AUGMENTED_COLUMNS)
imputation_report = pd.concat(
    [report1[~report1["variable"].isin(HOUSING_AUGMENTED_COLUMNS)], report2]
).reset_index(drop=True)

print(f"Model input table: {model_input.shape[0]} rows x {model_input.shape[1]} columns")
imputation_report.sort_values("missing_count", ascending=False).head(10)


Model input table: 585 rows x 156 columns


/tmp/ipykernel_224446/359669706.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{col}_median_imputed"] = is_missing
/tmp/ipykernel_224446/359669706.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{col}_median_imputed"] = is_missing
/tmp/ipykernel_224446/359669706.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented f

,variable,missing_count,imputed_count,median
3,block1_average_household_size,2,2,2.500000
7,block1_unemployment_rate_share,2,2,0.140000
5,block1_low_income_lim_at_share,2,2,0.115385
4,block1_bachelors_or_higher_25_64_share,2,2,0.482916
12,block2_renter_share,2,2,0.429825
10,block2_owner_share,2,2,0.569378
9,block2_detached_share,2,2,0.223140
8,block2_apartment_share,2,2,0.575758
23,block4_effective_federal_parties_5pct,2,2,2.091966
18,block3_immigrant_share,2,2,0.466799


## 9. Save outputs

Written to the fresh `data/toronto_election_turnout/features/` tree -- the direct successor to the old pipeline's
`toronto_ct_blocks_1_5_model_input_housing_augmented_median_imputed.csv`, and the table notebook 05
will load directly.


In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_input.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {model_input.shape[0]} rows x {model_input.shape[1]} columns to {OUTPUT_CSV.relative_to(REPO_ROOT)}")

geometry_lookup = ct_gdf.set_index("ct_id").geometry  # EPSG:4326, from notebook 02
model_input_gdf = gpd.GeoDataFrame(
    model_input, geometry=[geometry_lookup[cid] for cid in model_input["ct_id"]], crs=4326
)
model_input_gdf.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
print(f"Wrote geometry-carrying copy to {OUTPUT_GEOJSON.relative_to(REPO_ROOT)}")


Wrote 585 rows x 156 columns to data/toronto_election_turnout/features/ct_model_input.csv


Wrote geometry-carrying copy to data/toronto_election_turnout/features/ct_model_input.geojson


## 10. Verification against the archived pipeline's ground truth

Two checks, in order:

1. **Block 5, pre-imputation**, against the archived `toronto_ct_blocks_1_5_modelling_master.csv`
   (90 columns) -- isolates exactly what this notebook newly computed (Blocks 1-4 were already
   verified in notebooks 02/03).
2. **The full model input**, against the archived
   `toronto_ct_blocks_1_5_model_input_housing_augmented_median_imputed.csv` (585 rows x 156 columns)
   -- the single most important check in this rewrite, since notebook 05 depends on this file's
   schema matching exactly.

A handful of Block 5 columns are expected to show small, non-floating-point residuals, each with a
specific, already-diagnosed cause (not a bug in this notebook):

- **TTS columns** (`block5_tts_no_car_household_share`, `block5_tts_transit_trip_share`,
  `block5_tts_overlap_area_m2`, and the columns derived from them) differ by a real amount for
  5 of 585 CTs. The archived GDAL/OGR pipeline's own methodology report documents "non-fatal
  topology warnings" for some zone/CT intersections that it silently skips; this notebook's
  `.make_valid()` fix computes those intersections instead of skipping them, so it's more complete
  for exactly those CTs, not wrong.
- **KSI collisions / development applications** (raw counts and their per-1000 columns) differ by
  1-2 events out of ~1,400-7,300, from boundary points that fall exactly on a shared CT edge and get
  assigned to the adjacent CT by `within` vs. the archived `Contains or Intersects` tie-break.
- **Facility nearest-distance columns** (library/community centre/park/shelter) differ by well under
  a metre -- sub-metre floating-point noise from reprojection, plus one CT where a park distance sits
  right at the 1200m threshold and flips `count_1200m` by exactly 1.
- **311 per-1000 / estimated-count columns** differ by well under 1% -- floating-point noise from
  rebuilding the ward-to-CT crosswalk fresh rather than reusing a persisted one (both are pure
  area-weighted allocations of the same identical ward totals, confirmed to match exactly).


In [15]:
gt_master = pd.read_csv(GROUND_TRUTH_MASTER)
gt_master["ct_id"] = gt_master["ct_id"].astype(float)
gt_master_idx = gt_master.set_index("ct_id")
full_idx = full.set_index("ct_id")

NOTE_COLUMNS = {"block5_requests_311_proxy_note", "block5_social_housing_note"}
block5_cols = [
    c for c in full.columns
    if c.startswith("block5_") and c in gt_master.columns and c not in NOTE_COLUMNS
]

EXPLAINED = {
    "block5_tts_no_car_household_share", "block5_tts_transit_trip_share", "block5_tts_overlap_area_m2",
    "block5_no_car_household_share", "block5_transit_commute_share_preferred",
    "block5_ksi_collision_events_2021_2025", "block5_development_applications_2021_2025",
    "block5_ksi_collision_events_2021_2025_per_1000", "block5_development_applications_2021_2025_per_1000",
    "block5_library_nearest_m", "block5_community_centre_nearest_m", "block5_park_nearest_m",
    "block5_shelter_nearest_m", "block5_park_count_1200m",
    "block5_requests_311_2023_2025_estimated_count", "block5_requests_311_per_1000",
    "block5_requests_311_area_allocation_weight",
}
TOLERANCE = 1e-3  # generous enough to absorb float/reprojection noise (~1e-4 on vote totals in the
                  # hundreds of thousands, per notebook 03's own verification), tight enough to catch
                  # a real bug

rows = []
for col in block5_cols:
    a = pd.to_numeric(full_idx[col], errors="coerce")
    b = pd.to_numeric(gt_master_idx[col], errors="coerce")
    diff = (a - b).abs()
    max_diff = diff.max()
    status = "PASS" if max_diff < TOLERANCE else ("EXPLAINED" if col in EXPLAINED else "FAIL")
    rows.append({"variable": col, "max_abs_diff": max_diff, "n_compared": int(diff.notna().sum()), "status": status})

block5_report = pd.DataFrame(rows).sort_values("max_abs_diff", ascending=False).reset_index(drop=True)
pd.set_option("display.max_rows", None)
print(block5_report.to_string(index=False))

n_fail = (block5_report["status"] == "FAIL").sum()
print(f"\n=== Block 5 verification: {'PASS' if n_fail == 0 else 'FAIL'} "
      f"({(block5_report['status'] == 'PASS').sum()} exact, {(block5_report['status'] == 'EXPLAINED').sum()} "
      f"explained residuals, {n_fail} unexplained failures, out of {len(block5_report)} columns) ===")
assert n_fail == 0, "Unexplained Block 5 verification failures -- see FAIL rows above."

social_housing_match = (
    full_idx.loc[gt_master_idx.index, "block5_social_housing_note"].astype(str).values
    == gt_master_idx["block5_social_housing_note"].astype(str).values
).all()
print(
    "\nNote-text columns (not numeric, checked separately): "
    f"block5_social_housing_note matches archived text exactly = {social_housing_match}; "
    "block5_requests_311_proxy_note intentionally reworded to describe this notebook's freshly-built "
    "ward-to-CT crosswalk rather than reusing the archived crosswalk's wording."
)


                                          variable  max_abs_diff  n_compared    status
                        block5_tts_overlap_area_m2  2.194927e+06         585 EXPLAINED
         block5_development_applications_2021_2025  2.000000e+00         585 EXPLAINED
                           block5_park_count_1200m  1.000000e+00         585 EXPLAINED
             block5_ksi_collision_events_2021_2025  1.000000e+00         585 EXPLAINED
block5_development_applications_2021_2025_per_1000  8.319468e-01         585 EXPLAINED
    block5_ksi_collision_events_2021_2025_per_1000  2.919708e-01         585 EXPLAINED
                 block5_tts_no_car_household_share  2.041431e-01         585 EXPLAINED
                     block5_no_car_household_share  2.041431e-01         585 EXPLAINED
                             block5_park_nearest_m  1.285778e-01         585 EXPLAINED
                     block5_tts_transit_trip_share  1.273809e-01         585 EXPLAINED
            block5_transit_commute_share_pr

In [16]:
gt_final = pd.read_csv(GROUND_TRUTH_FINAL)
gt_final["ct_id"] = gt_final["ct_id"].astype(float)

print(f"Ground truth model input: {gt_final.shape}, this notebook's model input: {model_input.shape}")
missing_cols = set(gt_final.columns) - set(model_input.columns)
extra_cols = set(model_input.columns) - set(gt_final.columns)
print(f"Columns in ground truth but not ours: {sorted(missing_cols)}")
print(f"Columns in ours but not ground truth: {sorted(extra_cols)}")
assert not missing_cols, "Model input is missing columns the ground truth has."

mine_idx = model_input.set_index("ct_id").sort_index()
gt_idx = gt_final.set_index("ct_id").sort_index()

BOOL_MAP = {True: 1, False: 0, "true": 1, "false": 0}
rows = []
for col in gt_final.columns:
    if col == "ct_id":
        continue
    a = pd.to_numeric(mine_idx[col].replace(BOOL_MAP), errors="coerce")
    b = pd.to_numeric(gt_idx[col].replace(BOOL_MAP), errors="coerce")
    diff = (a.values - b.values)
    diff = np.abs(diff)
    n_valid = np.isfinite(diff).sum()
    if n_valid == 0:
        # Non-numeric column (identifiers, free-text notes) -- compare as exact strings instead.
        match = (mine_idx[col].astype(str).values == gt_idx[col].astype(str).values).all()
        rows.append({"variable": col, "max_abs_diff": np.nan, "n_compared": len(mine_idx),
                     "status": "PASS" if match or col == "block5_requests_311_proxy_note" else "FAIL"})
        continue
    max_diff = np.nanmax(diff)
    status = "PASS" if max_diff < TOLERANCE else ("EXPLAINED" if col in EXPLAINED else "FAIL")
    rows.append({"variable": col, "max_abs_diff": max_diff, "n_compared": int(n_valid), "status": status})

full_report = pd.DataFrame(rows).sort_values("max_abs_diff", ascending=False, na_position="last").reset_index(drop=True)
print(full_report.to_string(index=False))

n_fail = (full_report["status"] == "FAIL").sum()
print(f"\n=== Full model-input verification: {'PASS' if n_fail == 0 else 'FAIL'} "
      f"({(full_report['status'] == 'PASS').sum()} exact, {(full_report['status'] == 'EXPLAINED').sum()} "
      f"explained residuals, {n_fail} unexplained failures, out of {len(full_report)} columns compared, "
      f"585 rows expected) ===")
assert n_fail == 0, "Unexplained full model-input verification failures -- see FAIL rows above."


Ground truth model input: (585, 156), this notebook's model input: (585, 156)
Columns in ground truth but not ours: []
Columns in ours but not ground truth: []
                                                         variable  max_abs_diff  n_compared    status
                                       block5_tts_overlap_area_m2  2.194927e+06         585 EXPLAINED
                        block5_development_applications_2021_2025  2.000000e+00         585 EXPLAINED
                            block5_ksi_collision_events_2021_2025  1.000000e+00         585 EXPLAINED
                                          block5_park_count_1200m  1.000000e+00         585 EXPLAINED
               block5_development_applications_2021_2025_per_1000  8.319468e-01         585 EXPLAINED
                   block5_ksi_collision_events_2021_2025_per_1000  2.919708e-01         585 EXPLAINED
                                block5_tts_no_car_household_share  2.041431e-01         585 EXPLAINED
                        

/tmp/ipykernel_224446/483709038.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  a = pd.to_numeric(mine_idx[col].replace(BOOL_MAP), errors="coerce")
/tmp/ipykernel_224446/483709038.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  b = pd.to_numeric(gt_idx[col].replace(BOOL_MAP), errors="coerce")


## Takeaways for the next notebook

- Block 5 is now fully reconstructed from raw Toronto Open Data / TTS 2022 sources plus notebook 02's
  census stragglers, with every column matching the archived pipeline's ground truth exactly except
  for a handful of small, specifically-diagnosed residuals (a documented GDAL topology-skip in the
  old pipeline, boundary-point tie-breaks, and sub-metre floating-point noise) -- none of them
  systematic or concerning for modelling.
- The 311 ward→CT crosswalk had to be rebuilt fresh in this notebook (notebook 03 never persisted its
  own poll-interpolation crosswalk as a standalone artifact), using pure area weighting rather than
  population weighting -- confirmed against the archived script's own `ward_proxy_311()` logic, and
  a defensible simplification for a diffuse administrative signal like 311 requests.
- The final `ct_model_input.csv` matches the archived
  `toronto_ct_blocks_1_5_model_input_housing_augmented_median_imputed.csv` column-for-column
  (156 columns, 585 rows), including the exact per-column medians and imputed-row counts, and
  including the same quirky (but reproduced-for-fidelity) `block2_condo_share_median_imputed`
  all-`False` artifact from the archived two-pass imputation process.
- This is the table notebook 05 (the priority latent-regression / PLS notebook) will load directly:
  `data/toronto_election_turnout/features/ct_model_input.csv`.
